# Aula 07 · Unity Catalog: governança de dados

## O que é
Pense em uma **biblioteca**:

| Na biblioteca | No Unity Catalog |
|---|---|
| Prédio → andar → estante → livro | **Catálogo → schema → tabela** (`growdev.silver.customers`) |
| Depósito de caixas e documentos avulsos | **Volume** (`growdev.bronze.landing`) |
| Ficha de empréstimo: quem pode levar qual livro | **Permissões** (`GRANT`) |
| Anotação "este livro foi copiado daquele" | **Linhagem** (lineage) |

## Por que importa
Empresas exigem **controle de acesso e rastreabilidade**, como pede a LGPD. Os jobs do curso leem e escrevem em tabelas governadas pelo Unity Catalog.

## Como aplicar
### 1. Navegando no Catalog Explorer
Menu lateral → **Catalog** → `growdev`. Explore:
- `bronze` → **Tables** e **Volumes** → `landing`
- `gold` → `resumo_clientes` → abas **Overview**, **Sample Data**, **Permissions** e **Lineage**

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS growdev.analytics
COMMENT 'Tabelas compartilhadas com a área de análise';

CREATE OR REPLACE TABLE growdev.analytics.inadimplencia_por_segmento
COMMENT 'Taxa de inadimplência por segmento de cliente'
AS
SELECT customer_segment,
       SUM(qtd_emprestimos) AS emprestimos,
       SUM(qtd_inadimplencias) AS inadimplencias,
       ROUND(100 * SUM(qtd_inadimplencias) / SUM(qtd_emprestimos), 2) AS taxa_pct
FROM growdev.gold.resumo_clientes
GROUP BY customer_segment;

SELECT * FROM growdev.analytics.inadimplencia_por_segmento

### 3. Concedendo permissão a um grupo
Para alguém ler uma tabela, precisa de três "chaves": usar o catálogo, usar o schema e ler a tabela.

> No plano gratuito usamos o grupo `account users` (todos os usuários). Em uma empresa, você criaria um grupo como `analistas_credito` em **Settings → Identity and access → Groups**.

In [ ]:
%sql
GRANT USE CATALOG ON CATALOG growdev TO `account users`;
GRANT USE SCHEMA ON SCHEMA growdev.analytics TO `account users`;
GRANT SELECT ON TABLE growdev.analytics.inadimplencia_por_segmento TO `account users`;

SHOW GRANTS ON TABLE growdev.analytics.inadimplencia_por_segmento

In [ ]:
%sql
REVOKE SELECT ON TABLE growdev.analytics.inadimplencia_por_segmento FROM `account users`;
SHOW GRANTS ON TABLE growdev.analytics.inadimplencia_por_segmento

### 5. Linhagem
No Catalog Explorer, abra `growdev.analytics.inadimplencia_por_segmento` → aba **Lineage** → **See lineage graph**. Você verá o caminho completo:

```
landing → bronze.loans / bronze.customers → silver.loans / silver.customers → gold.resumo_clientes → analytics.inadimplencia_por_segmento
```

O Unity Catalog desenhou isso **sozinho**, só observando as leituras e escritas dos notebooks.

## Checklist da aula
- [ ] Sei o que é `catálogo.schema.tabela` e o que é um volume
- [ ] Criei um schema e uma tabela
- [ ] Concedi e revoguei permissão com `GRANT`/`REVOKE`
- [ ] Encontrei a linhagem de uma tabela no Catalog Explorer